In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision import transforms

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [3]:
transform = transforms.ToTensor()

train_dataset = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Train Samples :", len(train_dataset))
print("Test Samples  :", len(test_dataset))

Train Samples : 60000
Test Samples  : 10000


In [4]:
trainloader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

testloader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [5]:
class MLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Flatten(),

            nn.Linear(784,256),
            nn.ReLU(),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,10)

        )

    def forward(self,x):

        return self.network(x)

In [6]:
model = MLP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

MLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [28]:
def train(model, trainloader, testloader, epochs):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_accuracy = 0.0

    for epoch in range(epochs):

        # -----------------------
        # Training
        # -----------------------
        model.train()

        running_loss = 0

        for images, labels in trainloader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        # -----------------------
        # Testing
        # -----------------------
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in testloader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)

                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        print(f"Epoch {epoch+1}/{epochs} | Loss = {running_loss/len(trainloader):.4f} | Accuracy = {accuracy:.2f}%")

        # -----------------------
        # Save Best Model
        # -----------------------
        if accuracy > best_accuracy:

            best_accuracy = accuracy

            torch.save(model.state_dict(), "best_model_weights.pth")

            print(" Best Model Saved")
    return best_accuracy

In [29]:
best_acc = train(model, trainloader, testloader, epochs=5)

print(best_acc)

Epoch 1/5 | Loss = 0.0154 | Accuracy = 97.88%
 Best Model Saved
Epoch 2/5 | Loss = 0.0118 | Accuracy = 98.05%
 Best Model Saved
Epoch 3/5 | Loss = 0.0105 | Accuracy = 97.98%
Epoch 4/5 | Loss = 0.0099 | Accuracy = 97.88%
Epoch 5/5 | Loss = 0.0099 | Accuracy = 97.91%
98.05


In [9]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, pred = torch.max(outputs,1)

        total += labels.size(0)

        correct += (pred==labels).sum().item()

print("Current Model Accuracy =",100*correct/total)

Current Model Accuracy = 97.65


In [18]:
print("\n===== Model State Dictionary =====\n")

for key, value in model.state_dict().items():

    print(key, value.shape)


===== Model State Dictionary =====

network.1.weight torch.Size([256, 784])
network.1.bias torch.Size([256])
network.3.weight torch.Size([128, 256])
network.3.bias torch.Size([128])
network.5.weight torch.Size([10, 128])
network.5.bias torch.Size([10])


In [25]:
best_model = MLP().to(device)

state_dict = torch.load(
    "best_model_weights.pth",
    weights_only=True
)

best_model.load_state_dict(state_dict)

best_model.eval()

print("Best Model Loaded Successfully")

Best Model Loaded Successfully


In [20]:
print("\n===== Comparing Weights =====\n")

for key in model.state_dict():

    print(
        key,
        torch.equal(
            model.state_dict()[key],
            best_model.state_dict()[key]
        )
    )


===== Comparing Weights =====

network.1.weight True
network.1.bias True
network.3.weight True
network.3.bias True
network.5.weight True
network.5.bias True


In [21]:
images, labels = next(iter(testloader))

images = images.to(device)

model.eval()
best_model.eval()

with torch.no_grad():

    out1 = model(images)

    out2 = best_model(images)

print("Predictions Same :", torch.equal(out1, out2))

Predictions Same : True


In [22]:
correct = 0
total = 0

best_model.eval()

with torch.no_grad():

    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)

        _, pred = torch.max(outputs,1)

        total += labels.size(0)

        correct += (pred==labels).sum().item()

print("Loaded Model Accuracy =",100*correct/total)

Loaded Model Accuracy = 97.65


In [23]:
print("\n========== SUMMARY ==========")

print(f"Best Accuracy During Training : {best_acc:.2f}%")
print(f"Current Model Accuracy        : {100*correct/total:.2f}%")


========== SUMMARY ==========
Best Accuracy During Training : 97.65%
Current Model Accuracy        : 97.65%


In [24]:
current_correct = 0
current_total = 0

model.eval()

with torch.no_grad():

    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, pred = torch.max(outputs,1)

        current_total += labels.size(0)

        current_correct += (pred==labels).sum().item()

current_acc = 100 * current_correct / current_total


best_correct = 0
best_total = 0

best_model.eval()

with torch.no_grad():

    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = best_model(images)

        _, pred = torch.max(outputs,1)

        best_total += labels.size(0)

        best_correct += (pred==labels).sum().item()

best_acc_loaded = 100 * best_correct / best_total

print("\n========== FINAL RESULT ==========")

print(f"Best Accuracy During Training : {best_acc:.2f}%")
print(f"Current Model Accuracy        : {current_acc:.2f}%")
print(f"Loaded Model Accuracy         : {best_acc_loaded:.2f}%")


========== FINAL RESULT ==========
Best Accuracy During Training : 97.65%
Current Model Accuracy        : 97.65%
Loaded Model Accuracy         : 97.65%
